In [24]:
import requests
from dotenv import load_dotenv
import os
import json
from pathlib import Path

In [25]:
# load api key
load_dotenv()
tmdb_read_access_token = os.getenv(key='TMDB_API_READ_ACCESS_TOKEN')

In [26]:
popular_movies_url = 'https://api.themoviedb.org/3/movie/popular'
top_rated_movies_url = 'https://api.themoviedb.org/3/movie/top_rated'
now_playing_movies_url = 'https://api.themoviedb.org/3/movie/now_playing'
upcoming_movies_url = 'https://api.themoviedb.org/3/movie/upcoming'
genre_list_url = 'https://api.themoviedb.org/3/genre/movie/list'
language_list_url = 'https://api.themoviedb.org/3/configuration/languages'

headers = {
    "accept":"application/json",
    "Authorization": f"Bearer {tmdb_read_access_token}",
    "User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
}

# print(repr(tmdb_read_access_token))

In [18]:
try:
    params = {'page': 502}
    response = requests.get(url=upcoming_movies_url, headers=headers, params=params)
    data = response.json()
    # print(data)
    with open('data.json', 'w') as f:
        json.dump(data, f)
except Exception as e:
    print(f"Error while getting the data: {e}")

Error while getting the data: ('Connection aborted.', ConnectionResetError(54, 'Connection reset by peer'))


In [22]:
import asyncio
import aiohttp
import json

top_rated_movies = []
# page_count = 1

semaphore = asyncio.Semaphore(8)

async def fetch_page(session, url, page, attempt=1, max_attempts=5):
    async with semaphore:
        async with session.get(url, headers=headers, params={'page': page}) as resp:
            if resp.status == 429:
                if attempt > max_attempts:
                    raise RuntimeError(f"Exceeded max retries for page {page}")
                retry_after = int(resp.headers.get('Retry-After', 1))
                await asyncio.sleep(retry_after)
                return await fetch_page(session, url, page, attempt+1, max_attempts)  # retry
            return await resp.json()


async def main(url, path):
    async with aiohttp.ClientSession() as session:
        count = 1
        for batch_start in range(1, 501, 25):

            # skip the already written file
            batch_file = Path(path) / f"batch_{count}.json"
            if batch_file.exists():
                count += 1
                continue


            batch = range(batch_start, min(batch_start + 25, 501))
            tasks = [fetch_page(session, url, p) for p in batch]
            results = await asyncio.gather(*tasks, return_exceptions=True)

            successes = [result for result in results if not isinstance(result, Exception)]
            failures = [(page, result) for page, result in zip(batch, results) if isinstance(result, Exception)]

            if failures:
                print(f"Batch {batch_start}: failed pages {[p for p, _ in failures]}")

            # top_rated_movies.extend(results)
            with open(batch_file, 'w') as file:
                json.dump(successes, file)
            count += 1
            await asyncio.sleep(1)  # brief pause between batches


movie_categories = ['top_rated_movies', 'popular_movies']
for category in movie_categories:
    os.makedirs(category, exist_ok=True)


# await main(url=top_rated_movies_url, path='top_rated_movies')
# await main(url=popular_movies_url, path='popular_movies')
# await main(url=now_playing_movies_url, path='now_playing_movies')
# await main(url=upcoming_movies_url, path='upcoming_movies')
await main(url=genre_list_url, path='genre_list')

# async def fetch(url, session, headers, params):
#     async with session.get(url, headers=headers, params=params) as response:
#         return await response.json()
        
# async def main():

#     # to track the page count
#     page_count = 1

#     async with aiohttp.ClientSession() as session:

#         while True:

#             query_params = {'page': page_count}
#             # result = await fetch(url=top_rated_movies_url, session=session, headers=headers, params=query_params)

#             try:
#                 # adding timeout requests
#                 result = await asyncio.wait_for(fetch(url=top_rated_movies_url, session=session, headers=headers, params=query_params), timeout=10)
#             except Exception as e:
#                 print(f"Error on fecthing the data: {e}")
#                 continue

#             total_pages = result.get('total_pages')
#             if total_pages is None or page_count > total_pages:
#                 break


#             top_rated_movies.append(result)

#             page_count += 1

#             # delay between requests
#             await asyncio.sleep(0.5)


# await main(url=top_rated_movies_url)





# try:
#     response = requests.get(url=top_rated_movies_url, headers=headers)
#     data = response.json()
#     # print(data)
#     with open('top_rated_data.json', 'w') as f:
#         json.dump(data, f)
# except Exception as e:
#     print(f"Error while getting the data: {e}")

In [27]:
current_dir = Path.cwd()

In [28]:
output_dir = current_dir.parent / 'movie_pool' / 'genre_master' 
output_dir.mkdir(parents=True, exist_ok=True)
file_path = output_dir / 'genre_master.json'

response = requests.get('https://api.themoviedb.org/3/genre/movie/list', headers=headers)
data = response.json()
with open(file_path, 'w') as f:
    json.dump(data, f)


In [23]:
output_dir = current_dir.parent / 'movie_pool' / 'language_master' 
output_dir.mkdir(parents=True, exist_ok=True)
file_path = output_dir / 'language_master.json'

response = requests.get(language_list_url, headers=headers)
data = response.json()
with open(file_path, 'w') as f:
    json.dump(data, f)